<a href="https://colab.research.google.com/github/RobinJames25/APP2-FSTACK/blob/main/AI_recruitement_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 45.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 105.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pymupdf]


In [29]:
from google.colab import files

uploaded_files = files.upload()

Saving corrupted.pdf to corrupted (1).pdf


In [31]:
import pymupdf
import logging

logging.basicConfig(level=logging.ERROR)

def read_cv(file_path):
  try:
    with pymupdf.open(file_path) as document:

      if len(document) == 0:
        return {
            "status": "empty",
            "pages": [],
            "error": None
        }

      pages = []
      matrix = pymupdf.Matrix(2, 2)

      for page_number, page in enumerate(document, start=1):
        try:
          text_extracted = page.get_text()

          if text_extracted.strip():
            pages.append({
                "page_number": page_number,
                "content_type": "text",
                "processing_status": "completed",
                "content": text_extracted,
                "error": None
            })
          else:
            if not page.get_images() and not page.get_drawings():
              pages.append({
                  "page_number": page_number,
                  "processing_status": "completed",
                  "content_type": "blank",
                  "content": None,
                  "error": None
              })
            else:
              image_extracted = page.get_pixmap(matrix=matrix)
              pages.append({
                  "page_number": page_number,
                  "content_type": "image",
                  "processing_status": "needs_ocr",
                  "content": image_extracted,
                  "error": None
              })
        except Exception as error:
          logging.error(
              "Failed to process page %s: %s",
              page_number,
              error
          )
          pages.append({
              "page_number": page_number,
              "content_type": "error",
              "processing_status": "failed",
              "content": None,
              "error": str(error)
          })
    if any(page["processing_status"] == "failed" for page in pages):
        status = "partial"
    elif any(page["processing_status"] == "needs_ocr" for page in pages):
      status = "needs_ocr"
    else:
        status = "success"
    return {
      "status": status,
      "pages": pages,
      "error": None
    }
  except Exception as error:
    logging.error("Failed to read PDF: %s", error)
    return {
      "status": "failed",
      "pages": [],
      "error": str(error)
    }
if __name__ == '__main__':
  file_path = r"/content/corrupted.pdf"
  result = read_cv(file_path)
  print(result)
  if result["pages"]:
    for page in result["pages"]:
      print(page["content"])

ERROR:root:Failed to read PDF: Failed to open file '/content/corrupted.pdf' as type pdf.


{'status': 'failed', 'pages': [], 'error': "Failed to open file '/content/corrupted.pdf' as type pdf."}


In [9]:
import copy
# ==========================================
# 1. APPLICANT VALIDATION
# ==========================================

def check_requirements(applicant_id, name, education, years_of_experience, skills):

    if isinstance(skills, (list, tuple)):
        skills = list(skills)

    applicant_details = {
        "applicant_id": applicant_id,
        "name": name,
        "education": education,
        "years_of_experience": years_of_experience,
        "skills": skills,
        "processing_status": "pending_review",
        "missing_fields": [],
        "validation_errors": []
    }

    skills_are_valid_container = (
        skills is None or isinstance(skills, (list, tuple))
    )

    if not skills_are_valid_container:
        applicant_details["validation_errors"].append(
            "skills must be a list or tuple"
        )

    if not isinstance(applicant_details["applicant_id"], str):
        print("Applicant ID must be a string")
        applicant_details["processing_status"] = "invalid"
        return applicant_details

    applicant_details["applicant_id"] = (
        applicant_details["applicant_id"].strip()
    )

    if applicant_details["applicant_id"] == "":
        print("Applicant ID cannot be empty")
        applicant_details["processing_status"] = "invalid"
        return applicant_details

    for qualification in ("name", "education", "years_of_experience", "skills"):
        details = applicant_details[qualification]

        if qualification in ("name", "education"):
            if details is None or (
                isinstance(details, str) and details.strip() == ""
            ):
                applicant_details["missing_fields"].append(qualification)
                continue

            if not isinstance(details, str):
                applicant_details["validation_errors"].append(
                    f"{qualification} must be a string"
                )
                continue

        if qualification == "years_of_experience":
            if details is None:
                applicant_details["missing_fields"].append(qualification)
                continue

            if (
                not isinstance(details, int)
                or isinstance(details, bool)
                or details < 0
            ):
                applicant_details["validation_errors"].append(
                    "years_of_experience must be a non-negative integer"
                )

        if qualification == "skills":
            if not skills_are_valid_container:
                continue

            if details is None or len(details) == 0:
                applicant_details["missing_fields"].append(qualification)
                continue

            for index, item in enumerate(details):
                if not isinstance(item, str) or item.strip() == "":
                    applicant_details["validation_errors"].append(
                        f"{qualification} contains an invalid item"
                    )
                    continue

                item = item.strip().lower()
                details[index] = item

    if applicant_details["validation_errors"]:
        applicant_details["processing_status"] = "invalid"
    elif applicant_details["missing_fields"]:
        applicant_details["processing_status"] = "pending_review"
    else:
        applicant_details["processing_status"] = "ready"

    return applicant_details


# ==========================================
# 2. APPLICANT ELIGIBILITY
# ==========================================

def check_eligibility(applicant_details):

    eligibility = {
        "eligible": False,
        "status": "pending_review",
        "reasons": []
    }

    if not isinstance(applicant_details, dict):
        eligibility["reasons"].append(
            "Applicant details must be a dictionary"
        )
        return eligibility

    processing_status = applicant_details.get("processing_status")
    if processing_status == "invalid":
      eligibility["status"] = "invalid"
      eligibility["reasons"].append(
          "Applicant information failed validation"
      )
      return eligibility

    elif processing_status == "pending_review":
      eligibility["reasons"].append(
          "Applicant information is incomplete"
      )
      return eligibility

    elif processing_status != "ready":
      eligibility["reasons"].append(
          "Applicant has an unrecognized processing status"
      )
      return eligibility


    if "education" not in applicant_details:
      eligibility["reasons"].append("Education information is missing")

    if "years_of_experience" not in applicant_details:
      eligibility["reasons"].append("Years of experience is missing")

    if eligibility["reasons"]:
      return eligibility

    education = applicant_details.get("education")
    years = applicant_details.get("years_of_experience")

    permitted_education = [
        "bsc. software engineering",
    ]

    if years is None:
        eligibility["reasons"].append(
            "Years of experience is missing"
        )
    elif not isinstance(years, int) or isinstance(years, bool):
      eligibility["reasons"].append(
          "Years of experience must be an interger"
      )
    elif years < 0:
      eligibility["reasons"].append(
          "Experience cannot be negative"
      )

    if not isinstance(education, str):
      eligibility["reasons"].append(
            "Education must be a string"
        )

    elif education.strip() == "":
       eligibility["reasons"].append(
        "Education information is missing"
    )

    if eligibility["reasons"]:
      return eligibility

    normalized_education = education.strip().lower()

    if normalized_education not in permitted_education:
        eligibility["status"] = "ineligible"
        eligibility["reasons"].append(
            "Applicant does not meet the education requirement"
        )

    if years < 2:
        eligibility["status"] = "ineligible"
        eligibility["reasons"].append(
            "Applicant does not meet the minimum experience requirement"
        )

    if eligibility["status"] != "ineligible":
        eligibility["eligible"] = True
        eligibility["status"] = "eligible"

    return eligibility


# ==========================================
# 3. APPLICANT SCORING
# ==========================================

def calculate_scores(qualification):

    if not isinstance(qualification, dict):
        print("Error: This function expects a dictionary")
        return None

    if qualification.get("processing_status") != "ready":
        print("Cannot calculate scores. Applicant is not ready.")
        return None

    candidate_score = 0
    years = qualification.get("years_of_experience")

    if (
        not isinstance(years, int)
        or isinstance(years, bool)
        or years < 0
    ):
        print("Invalid years of experience.")
        return None

    candidate_score += years * 5

    skills = qualification.get("skills")

    if not isinstance(skills, (list, tuple)):
        print("Invalid skills data.")
        return None

    for skill in skills:
        if not isinstance(skill, str) or skill.strip() == "":
            print("Invalid skill item.")
            return None

    normalized_skills = [
        skill.strip().lower() for skill in skills
    ]

    unique_skills = set(normalized_skills)

    for skill in unique_skills:
        if skill == "python":
            candidate_score += 15
        elif skill == "sql":
            candidate_score += 15
        elif skill == "javascript":
            candidate_score += 10

    return candidate_score

def rank_applicants(applicants, top_n=5):
  if (
      not isinstance(top_n, int)
      or isinstance(top_n, bool)
      or top_n <= 0
  ):
    raise ValueError("top_n must be a positive integer")

  if not isinstance(applicants, list):
    raise ValueError("Applicants must be provided as a list")

  new_applicants = copy.deepcopy(applicants)
  batch_ids = set()
  scores = []
  not_proceeding = []

  for applicant in new_applicants:
    if not isinstance(applicant, dict):
      not_proceeding.append({
          "applicant": applicant,
          "eligibility": {
              "eligible": False,
              "status": "invalid",
              "reasons": ["Applicant record must be a dictionary"]
          },
          "reason": ["Applicant record must be a dictionary"]
      })
      continue

    applicant_id = applicant.get("applicant_id")
    if isinstance(applicant_id, str):
      applicant_id = applicant_id.strip()

      if applicant_id != "":
        if applicant_id in batch_ids:
          not_proceeding.append({
              "applicant": applicant,
              "eligibility": {
                  "eligible": False,
                  "status": "invalid",
                  "reasons": ["Duplicate applicant ID"]
              },
              "reason": ["Duplicate applicant ID"]
          })
          continue
        batch_ids.add(applicant_id)
    applicant_details = check_requirements(
        applicant.get("applicant_id"),
        applicant.get("name"),
        applicant.get("education"),
        applicant.get("years_of_experience"),
        applicant.get("skills")
      )
    eligibility = check_eligibility(applicant_details)
    if eligibility["eligible"] == True:
      candidate_score = calculate_scores(applicant_details)
      if candidate_score is not None:
        scores.append({
            "applicant": applicant_details,
            "score": candidate_score,
            "eligibility": eligibility
        })
      else:
        not_proceeding.append({
            "applicant": applicant_details,
            "eligibility": eligibility,
            "reason": ["Candidate score calculation failed"]
        })
    else:
      not_proceeding.append({
          "applicant": applicant_details,
          "eligibility": eligibility,
          "reason": eligibility["reasons"]
      })
  sorted_scores = sorted(
      scores,
      key=lambda item: item["score"],
      reverse=True
    )

  top_applicants = sorted_scores[:top_n]

  return {
      "top_applicants": top_applicants,
      "ranked_applicants": sorted_scores,
      "not_proceeding": not_proceeding
  }

